# Training on Google Colab

Colab is a **different computer** — it cannot see your local drive. So this notebook:

1. pulls the **code** in from GitHub
2. pulls the **data** in by upload
3. trains
4. pushes the **model** back out to your machine

## Read this before you start

**Colab's disk is temporary.** Everything under `/content/` is deleted when the
session ends — idle timeout, or ~12 hours maximum. If you train for 20 minutes
and close the tab without running the last cell, the model is gone.

**Enable the GPU first:** Runtime → Change runtime type → Hardware accelerator → GPU.
Do this *before* running anything; switching later restarts the session and
wipes your uploads.

## 1. Get the code

If the repo is private, this fails. Either make it public, or use the fallback
in the next cell (upload a zip of the repo instead).

In [ ]:
!git clone -b eavan-train-overall https://github.com/eavan127/sedicAI_NEXA.git
%cd sedicAI_NEXA
!ls

In [ ]:
# Colab already has torch, numpy, scipy, sklearn, matplotlib.
# Only these are missing:
!pip install -q pyyaml h5py

## 2. Check the GPU is actually attached

If this says `CUDA: False`, you skipped the Runtime → Change runtime type step.
Training still works on CPU, just slower.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 3. Get the data in

Upload the three arrays from your machine:

    data/processed/X.npy
    data/processed/y.npy
    data/processed/snr_labels.npy

About 29 MB total — seconds to upload.

**Upload the processed arrays, not RadChar.** RadChar is 400 MB and you would
only be rebuilding what you already built locally.

In [ ]:
import os, shutil
from google.colab import files

os.makedirs('data/processed', exist_ok=True)
print('Select X.npy, y.npy and snr_labels.npy (you can pick all three at once)')
uploaded = files.upload()

for name in uploaded:
    shutil.move(name, f'data/processed/{name}')

!ls -la data/processed/

### Alternative: mount Google Drive

Better if you will run this repeatedly — the files persist between sessions, so
you upload once instead of every time. Put the arrays in a `sedic/` folder in
your Drive first.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p data/processed
# !cp /content/drive/MyDrive/sedic/*.npy data/processed/

## 4. Sanity check before spending GPU time

Confirms the code runs and the data loaded correctly. Takes seconds. If this
fails, do not start the real run.

In [ ]:
!python -m pytest -q

import numpy as np
X = np.load('data/processed/X.npy')
y = np.load('data/processed/y.npy')
print('X:', X.shape, X.dtype)
print('class counts:', np.bincount(y))

## 5. Train

Watch `val_loss`. Falling means learning; rising while `train_loss` falls means
overfitting (harmless here — only the best checkpoint is kept).

In [ ]:
!python -m src.train

In [ ]:
!python -m src.evaluate

## 6. Get the model out — DO NOT SKIP THIS

This is the cell people forget. Everything above is deleted when the session
ends. `best_model.pt` **is** the trained model — a few MB of weights. Without
it, the code is an untrained shell and the run was wasted.

Save the downloads into `results/` and `evals/` in your local repo.

In [ ]:
from google.colab import files
import os

for path in ['results/best_model.pt',
             'evals/scorecard.json',
             'evals/confusion_matrix.png',
             'evals/accuracy_vs_snr.png']:
    if os.path.exists(path):
        files.download(path)
    else:
        print('missing:', path)

### Or save straight to Drive (survives the session)

In [ ]:
# !mkdir -p /content/drive/MyDrive/sedic/runs
# !cp results/best_model.pt evals/*.json evals/*.png /content/drive/MyDrive/sedic/runs/